# sugar-beets → 350×350 确定性切块（写入 `EWS/data/pretrain/npy`）

- **数据**：`EWS/data/sugar-beets-2016-DatasetNinja/ds/img/` 下 `rgb_*.png`（宽 1296、高 966；`np.asarray` 为 `H=966, W=1296`）。
- **策略**：不使用 AR-SSL4M 式随机起点；按 **最少** 个 **350×350** patch **无重叠** 铺满全图：
  - 行数 \(n_y=\lceil 966/350\rceil=3\)，列数 \(n_x=\lceil 1296/350\rceil=4\)，每图 **12** 块。
  - 画布补至 **1050×1400**（仅 **下、右** 补零，保留原图完整像素）。
  - 第 \(r\) 行、第 \(c\) 列块对应原画布坐标：\(y\in[r\cdot350,(r+1)\cdot350)\)，\(x\in[c\cdot350,(c+1)\cdot350)\)。
- **输出**：每个 patch 一个 `.npy`，**直接**写入已存在的 **`EWS/data/pretrain/npy/`**（不再套子目录；**不会 `mkdir`**，目录须事先建好）。
- **数组**：`float32`，形状 `(350, 350, 3)`，数值范围 \([0,1]\)（由 uint8 除以 255）。
- **进度**：最后一格使用 `tqdm` 总进度条；**每处理完 100 张图**打印一行阶段性提示；若环境无 `tqdm`，可执行 `pip install tqdm`。

In [8]:
from pathlib import Path

import numpy as np
from PIL import Image

PATCH = 350
H0, W0 = 966, 1296  # sugar-beets rgb 固定尺寸
N_ROWS = (H0 + PATCH - 1) // PATCH  # 3
N_COLS = (W0 + PATCH - 1) // PATCH  # 4
CANVAS_H = N_ROWS * PATCH  # 1050
CANVAS_W = N_COLS * PATCH  # 1400
PAD_BOTTOM = CANVAS_H - H0
PAD_RIGHT = CANVAS_W - W0

print(
    f"网格: {N_ROWS}×{N_COLS} = {N_ROWS * N_COLS} patches / 图; "
    f"画布 {CANVAS_H}×{CANVAS_W}; pad下={PAD_BOTTOM}, 右={PAD_RIGHT}"
)

网格: 3×4 = 12 patches / 图; 画布 1050×1400; pad下=84, 右=104


In [ ]:
def find_repo(start: Path) -> Path:
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / "EWS" / "data" / "sugar-beets-2016-DatasetNinja" / "ds" / "img").is_dir():
            return p
    raise FileNotFoundError(
        "未找到数据集目录。请在仓库根 AR-SSL4M 下启动内核，或把 cwd 切到该仓库。"
    )


REPO = find_repo(Path.cwd())
IMG_DIR = REPO / "EWS" / "data" / "sugar-beets-2016-DatasetNinja" / "ds" / "img"
OUT_DIR = REPO / "EWS" / "data" / "pretrain" / "npy"
if not OUT_DIR.is_dir():
    raise FileNotFoundError(
        f"输出目录必须已存在（不自动创建），请将 .npy 放在: {OUT_DIR}"
    )

print("REPO:", REPO)
print("IMG_DIR:", IMG_DIR)
print("OUT_DIR:", OUT_DIR)

In [ ]:
def image_to_padded_canvas(arr_hwc: np.ndarray) -> np.ndarray:
    """arr: (H0, W0, C) uint8，左上对齐，下/右补零至画布。"""
    if arr_hwc.shape[0] != H0 or arr_hwc.shape[1] != W0:
        raise ValueError(f"期望 H,W = {H0},{W0}, 得到 {arr_hwc.shape[:2]}")
    return np.pad(
        arr_hwc,
        ((0, PAD_BOTTOM), (0, PAD_RIGHT), (0, 0)),
        mode="constant",
        constant_values=0,
    )


def save_patches_for_image(img_path: Path) -> list[Path]:
    stem = img_path.stem
    im = Image.open(img_path).convert("RGB")
    arr = np.asarray(im, dtype=np.uint8)
    canvas = image_to_padded_canvas(arr)
    saved = []
    for r in range(N_ROWS):
        for c in range(N_COLS):
            y0, y1 = r * PATCH, (r + 1) * PATCH
            x0, x1 = c * PATCH, (c + 1) * PATCH
            patch = canvas[y0:y1, x0:x1, :].astype(np.float32) / 255.0
            out_path = OUT_DIR / f"{stem}__r{r}_c{c}.npy"
            np.save(out_path, patch, allow_pickle=False)
            saved.append(out_path)
    return saved

In [ ]:
pip install tqdm

In [7]:
from tqdm.auto import tqdm

rgb_paths = sorted(IMG_DIR.glob("rgb_*.png"))
print(f"共 {len(rgb_paths)} 张 rgb 图，每张导出 {N_ROWS * N_COLS} 个 npy")

all_saved = []
for i, p in enumerate(tqdm(rgb_paths, desc="切分 sugar-beets", unit="图")):
    paths = save_patches_for_image(p)
    all_saved.extend(paths)
    if (i + 1) % 100 == 0:
        tqdm.write(
            f"已完成 {i + 1} 张图（本批末张 {p.name}），累计 {len(all_saved)} 个 npy"
        )

print(f"全部完成，共写入 {len(all_saved)} 个 .npy（均在 {OUT_DIR} 下）")

共 12714 张 rgb 图，每张导出 12 个 npy


切分 sugar-beets:   1%|          | 102/12714 [00:03<07:27, 28.20图/s]  

已完成 100 张图（本批末张 rgb_1461247086_540069954.png），累计 1200 个 npy


切分 sugar-beets:   2%|▏         | 204/12714 [00:07<08:37, 24.20图/s]   

已完成 200 张图（本批末张 rgb_1461247186_541445788.png），累计 2400 个 npy


...

已完成 12600 张图（本批末张 rgb_bonirob_2016-06-14-12-07-37_4_frame107.png），累计 151200 个 npy


切分 sugar-beets: 100%|█████████▉| 12704/12714 [29:23<00:00, 12.17图/s]   

已完成 12700 张图（本批末张 rgb_bonirob_2016-06-14-12-07-37_4_frame83.png），累计 152400 个 npy


切分 sugar-beets: 100%|██████████| 12714/12714 [29:23<00:00,  7.21图/s]

全部完成，共写入 152568 个 .npy（均在 D:\Cursor\AR-SSL4M\pretrain\npy 下）
